# LeetCode 1319 — Number of Operations to Make Network Connected bằng Python

Link đề: https://leetcode.com/problems/number-of-operations-to-make-network-connected/ · Độ khó: **Medium** · Chủ đề: Depth-First Search, Breadth-First Search, Union-Find, Graph Theory

Skeleton LeetCode:

```python
class Solution:
    def makeConnected(self, n: int, connections: list[list[int]]) -> int:
        ...
```

## 1. Mục tiêu học

1. Mô hình hoá bài toán "mạng máy tính + dây cáp" thành **đồ thị vô hướng**: đỉnh = máy, cạnh = dây.
2. Hiểu hai sự thật then chốt: (a) nối `n` đỉnh cần **ít nhất `n − 1` cạnh**; (b) nếu đồ thị có `c` **thành phần liên thông** thì cần đúng `c − 1` lần di chuyển dây.
3. Đếm thành phần liên thông bằng ba công cụ kinh điển: **DFS (ngăn xếp)**, **BFS (hàng đợi)** và **Union-Find (DSU)** có nén đường đi.
4. Biết dùng **brute force trên không gian trạng thái** (BFS trên các đồ thị) với `n` rất nhỏ để kiểm chứng công thức.
5. Tránh lỗi: quên điều kiện `m < n − 1`, đệ quy quá sâu, `union` sai (gán cha cho đỉnh không phải gốc), dùng công thức `n − m` sai chỗ.

## 2. Đề bài & ví dụ

Có `n` máy tính đánh số `0..n−1`, nối với nhau bằng các dây `connections[i] = [a, b]` (dây nối trực tiếp máy `a` và `b`). Một **thao tác**: rút một dây đang nối hai máy và cắm lại vào **một cặp máy chưa nối trực tiếp**.

Trả về **số thao tác ít nhất** để mọi máy đều liên lạc được với nhau (trực tiếp hoặc gián tiếp). Nếu không thể, trả về `-1`.

**Ví dụ 1:** `n = 4, connections = [[0,1],[0,2],[1,2]]` → `1` (rút dây 1–2, cắm vào 1–3).

**Ví dụ 2:** `n = 6, connections = [[0,1],[0,2],[0,3],[1,2],[1,3]]` → `2`.

**Ví dụ 3:** `n = 6, connections = [[0,1],[0,2],[0,3],[1,2]]` → `-1` (không đủ dây).

**Ràng buộc:**
- `1 <= n <= 10^5`
- `1 <= connections.length <= min(n·(n−1)/2, 10^5)`
- `0 <= a, b < n`, `a != b`, không có dây lặp lại.

In [1]:
import random
import sys
import time
from collections import defaultdict, deque
from itertools import combinations
from typing import List

examples = [
    (4, [[0, 1], [0, 2], [1, 2]], 1),
    (6, [[0, 1], [0, 2], [0, 3], [1, 2], [1, 3]], 2),
    (6, [[0, 1], [0, 2], [0, 3], [1, 2]], -1),
]
for n, conns, want in examples:
    print(f"n={n}, số dây={len(conns)}, connections={conns} -> mong đợi {want}")

n=4, số dây=3, connections=[[0, 1], [0, 2], [1, 2]] -> mong đợi 1
n=6, số dây=5, connections=[[0, 1], [0, 2], [0, 3], [1, 2], [1, 3]] -> mong đợi 2
n=6, số dây=4, connections=[[0, 1], [0, 2], [0, 3], [1, 2]] -> mong đợi -1


## 3. Kiến thức nền

### 3.1 Danh sách kề (adjacency list)

Lưu đồ thị bằng `graph[u] = [các đỉnh kề u]`. Vì dây **vô hướng**, mỗi dây `[a, b]` phải thêm **hai chiều**: `b` vào `graph[a]` và `a` vào `graph[b]`. Có thể dùng `list` các `list` (đỉnh đánh số `0..n−1`) hoặc `defaultdict(list)`.

In [2]:
n = 6
conns = [[0, 1], [0, 2], [0, 3], [1, 2]]
graph = [[] for _ in range(n)]
for a, b in conns:
    graph[a].append(b)
    graph[b].append(a)
for u in range(n):
    print(u, "->", graph[u])

0 -> [1, 2, 3]
1 -> [0, 2]
2 -> [0, 1]
3 -> [0]
4 -> []
5 -> []


### 3.2 DFS bằng ngăn xếp (không đệ quy)

DFS = đi sâu nhất có thể rồi quay lui. Dùng `list` làm ngăn xếp: `append` để đẩy vào, `pop()` để lấy ra phần tử **cuối**. Mảng `seen` đánh dấu đỉnh đã thăm để không đi vòng mãi. Viết bằng ngăn xếp giúp tránh lỗi đệ quy quá sâu trong Python.

In [3]:
def dfs_order(graph, start, seen):
    order = []
    stack = [start]
    seen[start] = True
    while stack:
        u = stack.pop()
        order.append(u)
        for v in graph[u]:
            if not seen[v]:
                seen[v] = True
                stack.append(v)
    return order

seen = [False] * n
print("DFS từ 0:", dfs_order(graph, 0, seen))
print("seen sau DFS:", seen)

DFS từ 0: [0, 3, 2, 1]
seen sau DFS: [True, True, True, True, False, False]


### 3.3 BFS bằng `collections.deque`

BFS = loang theo từng lớp. `deque` cho `append` và `popleft` đều `O(1)` (còn `list.pop(0)` là `O(n)` — chậm).

In [4]:
def bfs_order(graph, start, seen):
    order = []
    q = deque([start])
    seen[start] = True
    while q:
        u = q.popleft()
        order.append(u)
        for v in graph[u]:
            if not seen[v]:
                seen[v] = True
                q.append(v)
    return order

seen = [False] * n
print("BFS từ 0:", bfs_order(graph, 0, seen))
print("BFS từ 4:", bfs_order(graph, 4, seen))

BFS từ 0: [0, 1, 2, 3]
BFS từ 4: [4]


### 3.4 Union-Find (Disjoint Set Union)

Mỗi tập được đại diện bởi một **gốc**. `parent[x]` là cha của `x`; gốc có `parent[x] == x`.

- `find(x)`: leo lên đến gốc. **Nén đường đi** (path compression): trên đường leo, gán thẳng cha của các đỉnh lên gần gốc ⇒ lần sau nhanh hơn.
- `union(a, b)`: tìm gốc `ra`, `rb`; nếu khác nhau thì gán `parent[ra] = rb` (gộp hai tập) và trả về `True`; nếu trùng ⇒ `a`, `b` đã cùng tập ⇒ dây này **dư thừa**.

In [5]:
parent = list(range(6))

def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]   # nén đường đi (halving)
        x = parent[x]
    return x

def union(a, b):
    ra, rb = find(a), find(b)
    if ra == rb:
        return False
    parent[ra] = rb
    return True

for a, b in [[0, 1], [0, 2], [0, 3], [1, 2]]:
    print(f"union({a},{b}) -> {union(a, b)}  parent={parent}")
print("số gốc =", sum(1 for i in range(6) if find(i) == i))

union(0,1) -> True  parent=[1, 1, 2, 3, 4, 5]
union(0,2) -> True  parent=[1, 2, 2, 3, 4, 5]
union(0,3) -> True  parent=[2, 2, 3, 3, 4, 5]
union(1,2) -> False  parent=[2, 3, 3, 3, 4, 5]
số gốc = 3


### 3.5 `itertools.combinations` — liệt kê mọi cặp

`combinations(range(n), 2)` sinh mọi cặp `(i, j)` với `i < j`. Ta sẽ dùng nó trong brute force để biết "những vị trí có thể cắm dây".

In [6]:
print(list(combinations(range(4), 2)))
print("số cặp với n=5:", len(list(combinations(range(5), 2))))

[(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
số cặp với n=5: 10


## 4. Ý tưởng nền tảng / trực giác

Tính tay **Ví dụ 2**: `n = 6`, dây `0-1, 0-2, 0-3, 1-2, 1-3`.

1. Vẽ ra: các máy `{0,1,2,3}` dính nhau; máy `4` đứng riêng; máy `5` đứng riêng ⇒ **3 thành phần liên thông**.
2. Để nối 3 "cụm" thành 1, cần ít nhất **2 dây nối giữa các cụm** (giống nối 3 hòn đảo bằng cầu: cần 2 cây cầu).
3. Lấy dây ở đâu? Cụm `{0,1,2,3}` có 4 đỉnh nhưng 5 dây; chỉ cần 3 dây là đủ giữ nó liên thông (một cây khung) ⇒ có **5 − 3 = 2 dây dư**. Đủ để làm 2 cây cầu ⇒ đáp án `2` ✔.

**Ví dụ 3**: chỉ còn 4 dây cho 6 máy. Muốn 6 máy liên thông cần tối thiểu `6 − 1 = 5` dây ⇒ không thể ⇒ `-1` ✔.

**Ví dụ 1**: `n = 4`, 3 dây tạo tam giác `0-1-2`, máy `3` riêng ⇒ 2 cụm; tam giác có 1 dây dư ⇒ đáp án `1` ✔.

In [7]:
def components_by_hand(n, conns):
    graph = [[] for _ in range(n)]
    for a, b in conns:
        graph[a].append(b)
        graph[b].append(a)
    seen = [False] * n
    comps = []
    for s in range(n):
        if not seen[s]:
            comps.append(sorted(dfs_order(graph, s, seen)))
    return comps

for n, conns, want in examples:
    comps = components_by_hand(n, conns)
    print(f"n={n}: cụm={comps}, số dây={len(conns)}, cần ít nhất {n-1} dây -> đáp án {want}")

n=4: cụm=[[0, 1, 2], [3]], số dây=3, cần ít nhất 3 dây -> đáp án 1
n=6: cụm=[[0, 1, 2, 3], [4], [5]], số dây=5, cần ít nhất 5 dây -> đáp án 2
n=6: cụm=[[0, 1, 2, 3], [4], [5]], số dây=4, cần ít nhất 5 dây -> đáp án -1


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** mô phỏng đúng như đề với `n` rất nhỏ. Một **trạng thái** là tập các dây hiện có. Từ một trạng thái, một thao tác = bỏ một dây đang có + thêm một cặp chưa có. Dùng **BFS trên không gian trạng thái**: số bước BFS đầu tiên đến được trạng thái liên thông chính là số thao tác ít nhất.

Vì số dây không đổi, nếu `m < n − 1` thì không trạng thái nào liên thông ⇒ BFS kết thúc mà không tìm thấy ⇒ `-1`.

In [8]:
def is_connected(n, edges):
    graph = [[] for _ in range(n)]
    for a, b in edges:
        graph[a].append(b)
        graph[b].append(a)
    seen = [False] * n
    return len(dfs_order(graph, 0, seen)) == n

def make_connected_brute(n, connections):
    start = frozenset(tuple(sorted(e)) for e in connections)
    all_pairs = list(combinations(range(n), 2))
    dist = {start: 0}
    q = deque([start])
    while q:
        state = q.popleft()
        if is_connected(n, state):
            return dist[state]
        for old in state:
            for new in all_pairs:
                if new not in state:
                    nxt = (state - {old}) | {new}
                    if nxt not in dist:
                        dist[nxt] = dist[state] + 1
                        q.append(nxt)
    return -1

for n, conns, want in examples[:1] + [(4, [[0, 1], [2, 3], [0, 2]], 0), (5, [[0, 1], [0, 2], [1, 2], [3, 4]], 1), (5, [[0, 1], [1, 2], [3, 4]], -1)]:
    print(n, conns, "->", make_connected_brute(n, conns), "| mong đợi", want)

4 [[0, 1], [0, 2], [1, 2]] -> 1 | mong đợi 1
4 [[0, 1], [2, 3], [0, 2]] -> 0 | mong đợi 0
5 [[0, 1], [0, 2], [1, 2], [3, 4]] -> 1 | mong đợi 1
5 [[0, 1], [1, 2], [3, 4]] -> -1 | mong đợi -1


**Độ phức tạp:** số trạng thái là `C(n(n−1)/2, m)` — bùng nổ tổ hợp. Với `n = 5` có 10 cặp, tối đa `C(10,5) = 252` trạng thái (ổn); với `n = 8` đã có 28 cặp, `C(28,14) ≈ 4·10^7` — không chạy nổi.

**Nút thắt:** ta thử mù quáng mọi cách rút/cắm, trong khi cấu trúc thật sự quan trọng chỉ là **số cụm** và **số dây**. Brute force vẫn rất có giá trị: nó là "trọng tài" để kiểm chứng công thức ở mục 12. (Ví dụ 2, 3 có `n = 6` hơi lớn cho brute force nên ta chỉ chạy ví dụ 1 và vài ví dụ nhỏ tự dựng.)

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — điều kiện cần:** một đồ thị liên thông `n` đỉnh có ít nhất `n − 1` cạnh (một cây có đúng `n − 1` cạnh). Thao tác không làm thay đổi số dây ⇒ **nếu `m < n − 1` thì trả về `-1`**.

**Insight 2 — số thao tác = số cụm − 1:**
- Mỗi thao tác thêm đúng 1 cạnh mới ⇒ giảm số cụm **tối đa 1** ⇒ cần **ít nhất `c − 1`** thao tác.
- Ngược lại, nếu `m ≥ n − 1` thì luôn đủ dây dư: một cụm có `k` đỉnh chỉ cần `k − 1` dây; tổng số dây "cần giữ" là `Σ(k_i − 1) = n − c`; số dây dư = `m − (n − c) ≥ (n − 1) − n + c = c − 1`. Mỗi dây dư rút ra **không làm vỡ cụm nào**, cắm nối hai cụm khác nhau ⇒ **đủ `c − 1` thao tác**.

**Quy tắc "nếu… thì…":**
- Nếu `len(connections) < n − 1` ⇒ `-1`.
- Ngược lại ⇒ đếm số thành phần liên thông `c` ⇒ trả `c − 1`.

Bài toán khó bỗng trở thành bài **đếm thành phần liên thông** — dùng DFS, BFS hoặc Union-Find.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — DFS đếm thành phần liên thông

**Từng đoạn:**
1. Kiểm tra `len(connections) < n − 1` ⇒ `-1` (làm **trước** để tiết kiệm thời gian).
2. Dựng danh sách kề hai chiều.
3. Duyệt từng đỉnh `s`; nếu `s` chưa thăm ⇒ phát hiện **cụm mới** ⇒ tăng `comps`, rồi DFS (ngăn xếp) đánh dấu toàn bộ cụm đó.
4. Trả về `comps − 1`.

In [9]:
def make_connected_dfs(n, connections):
    if len(connections) < n - 1:
        return -1
    graph = [[] for _ in range(n)]
    for a, b in connections:
        graph[a].append(b)
        graph[b].append(a)
    seen = [False] * n
    comps = 0
    for s in range(n):
        if seen[s]:
            continue
        comps += 1
        seen[s] = True
        stack = [s]
        while stack:
            u = stack.pop()
            for v in graph[u]:
                if not seen[v]:
                    seen[v] = True
                    stack.append(v)
    return comps - 1

for n, conns, want in examples:
    print(make_connected_dfs(n, conns), "| mong đợi", want)

1 | mong đợi 1
2 | mong đợi 2
-1 | mong đợi -1


Code nộp LeetCode (Cách 2):

In [10]:
class SolutionDFS:
    def makeConnected(self, n: int, connections: list[list[int]]) -> int:
        if len(connections) < n - 1:
            return -1
        graph = [[] for _ in range(n)]
        for a, b in connections:
            graph[a].append(b)
            graph[b].append(a)
        seen = [False] * n
        comps = 0
        for s in range(n):
            if seen[s]:
                continue
            comps += 1
            seen[s] = True
            stack = [s]
            while stack:
                u = stack.pop()
                for v in graph[u]:
                    if not seen[v]:
                        seen[v] = True
                        stack.append(v)
        return comps - 1

print([SolutionDFS().makeConnected(n, c) for n, c, _ in examples])

[1, 2, -1]


**Độ phức tạp:** `O(n + m)` thời gian (mỗi đỉnh vào ngăn xếp 1 lần, mỗi cạnh được xét 2 lần), `O(n + m)` bộ nhớ cho danh sách kề.

### 7.2 Cách 3 — BFS đếm thành phần liên thông

Giống hệt Cách 2, chỉ thay ngăn xếp bằng `deque` + `popleft`. Thứ tự thăm khác nhưng **tập đỉnh** của mỗi cụm như nhau ⇒ số cụm như nhau. Ở đây ta dùng thêm biến đếm `visited_count` để minh hoạ cách "dừng sớm": khi đã thăm đủ `n` đỉnh thì không cần quét tiếp.

In [11]:
def make_connected_bfs(n, connections):
    if len(connections) < n - 1:
        return -1
    graph = defaultdict(list)
    for a, b in connections:
        graph[a].append(b)
        graph[b].append(a)
    seen = [False] * n
    comps = 0
    visited_count = 0
    for s in range(n):
        if seen[s]:
            continue
        comps += 1
        seen[s] = True
        q = deque([s])
        while q:
            u = q.popleft()
            visited_count += 1
            for v in graph[u]:
                if not seen[v]:
                    seen[v] = True
                    q.append(v)
        if visited_count == n:
            break
    return comps - 1

for n, conns, want in examples:
    print(make_connected_bfs(n, conns), "| mong đợi", want)

1 | mong đợi 1
2 | mong đợi 2
-1 | mong đợi -1


Code nộp LeetCode (Cách 3):

In [12]:
class SolutionBFS:
    def makeConnected(self, n: int, connections: list[list[int]]) -> int:
        if len(connections) < n - 1:
            return -1
        graph = [[] for _ in range(n)]
        for a, b in connections:
            graph[a].append(b)
            graph[b].append(a)
        seen = [False] * n
        comps = 0
        for s in range(n):
            if seen[s]:
                continue
            comps += 1
            seen[s] = True
            q = deque([s])
            while q:
                u = q.popleft()
                for v in graph[u]:
                    if not seen[v]:
                        seen[v] = True
                        q.append(v)
        return comps - 1

print([SolutionBFS().makeConnected(n, c) for n, c, _ in examples])

[1, 2, -1]


**Độ phức tạp:** `O(n + m)` thời gian, `O(n + m)` bộ nhớ.

### 7.3 Cách 4 — Union-Find (không cần dựng đồ thị)

**Ý tưởng:** bắt đầu với `n` tập đơn lẻ (`comps = n`). Duyệt từng dây `[a, b]`:
- nếu `a`, `b` ở **hai tập khác nhau** ⇒ gộp lại, `comps -= 1`;
- nếu **cùng tập** ⇒ dây này dư, `extra += 1`.

Cuối cùng: nếu `extra < comps − 1` ⇒ `-1`, ngược lại ⇒ `comps − 1`. (Điều kiện này **tương đương** `m < n − 1`, vì `m = (n − comps) + extra`.) Ưu điểm: không cần danh sách kề, đọc từng cạnh một lần — rất hợp khi cạnh đến dạng "luồng".

In [13]:
def make_connected_uf(n, connections):
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    comps, extra = n, 0
    for a, b in connections:
        ra, rb = find(a), find(b)
        if ra == rb:
            extra += 1
        else:
            parent[ra] = rb
            comps -= 1
    return comps - 1 if extra >= comps - 1 else -1

for n, conns, want in examples:
    print(make_connected_uf(n, conns), "| mong đợi", want)

1 | mong đợi 1
2 | mong đợi 2
-1 | mong đợi -1


Phiên bản đầy đủ hơn dùng **union by size** (gộp cây nhỏ vào cây lớn) để đảm bảo độ cao cây nhỏ, kết hợp nén đường đi ⇒ mỗi thao tác gần như `O(1)` (chính xác là `O(α(n))`, `α` là hàm Ackermann ngược, ≤ 4 với mọi `n` thực tế).

In [14]:
class DSU:
    def __init__(self, n):
        self.parent = list(range(n))
        self.size = [1] * n
        self.count = n           # số tập hiện tại

    def find(self, x):
        root = x
        while self.parent[root] != root:
            root = self.parent[root]
        while self.parent[x] != root:          # nén đường đi hoàn toàn
            self.parent[x], x = root, self.parent[x]
        return root

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False
        if self.size[ra] < self.size[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        self.size[ra] += self.size[rb]
        self.count -= 1
        return True

d = DSU(6)
for a, b in examples[1][1]:
    print(f"union({a},{b}) = {d.union(a, b)}, count={d.count}, size={d.size}")

union(0,1) = True, count=5, size=[2, 1, 1, 1, 1, 1]
union(0,2) = True, count=4, size=[3, 1, 1, 1, 1, 1]
union(0,3) = True, count=3, size=[4, 1, 1, 1, 1, 1]
union(1,2) = False, count=3, size=[4, 1, 1, 1, 1, 1]
union(1,3) = False, count=3, size=[4, 1, 1, 1, 1, 1]


Code nộp LeetCode (Cách 4):

In [15]:
class SolutionUF:
    def makeConnected(self, n: int, connections: list[list[int]]) -> int:
        if len(connections) < n - 1:
            return -1
        parent = list(range(n))

        def find(x: int) -> int:
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x

        comps = n
        for a, b in connections:
            ra, rb = find(a), find(b)
            if ra != rb:
                parent[ra] = rb
                comps -= 1
        return comps - 1

print([SolutionUF().makeConnected(n, c) for n, c, _ in examples])

[1, 2, -1]


**Độ phức tạp:** `O(n + m·α(n))` ≈ `O(n + m)` thời gian, `O(n)` bộ nhớ (chỉ mảng `parent`) — tiết kiệm hơn DFS/BFS vì không lưu danh sách kề.

## 8. TRACE từng bước

Trace Union-Find trên **Ví dụ 2** (`n = 6`): bảng trạng thái (mỗi dòng sau khi xử lý một dây).

| bước | dây | find(a) | find(b) | hành động | comps | extra |
|---|---|---|---|---|---|---|
| 0 | — | — | — | khởi tạo | 6 | 0 |
| 1 | 0-1 | 0 | 1 | gộp | 5 | 0 |
| 2 | 0-2 | 1 | 2 | gộp | 4 | 0 |
| 3 | 0-3 | 2 | 3 | gộp | 3 | 0 |
| 4 | 1-2 | 3 | 3 | dư | 3 | 1 |
| 5 | 1-3 | 3 | 3 | dư | 3 | 2 |

Kết thúc: `comps = 3`, `extra = 2 ≥ comps − 1 = 2` ⇒ đáp án `2`. Hàm dưới đây in đúng các biến này.

In [16]:
def trace_uf(n, connections):
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    comps, extra = n, 0
    print(f"khởi tạo: parent={parent}, comps={comps}, extra={extra}")
    for step, (a, b) in enumerate(connections, 1):
        ra, rb = find(a), find(b)
        if ra == rb:
            extra += 1
            act = "dư"
        else:
            parent[ra] = rb
            comps -= 1
            act = "gộp"
        print(f"bước {step}: dây {a}-{b} | find={ra},{rb} | {act:3} | parent={parent} comps={comps} extra={extra}")
    ans = comps - 1 if extra >= comps - 1 else -1
    print("=> đáp án", ans)
    return ans

trace_uf(6, [[0, 1], [0, 2], [0, 3], [1, 2], [1, 3]])

khởi tạo: parent=[0, 1, 2, 3, 4, 5], comps=6, extra=0
bước 1: dây 0-1 | find=0,1 | gộp | parent=[1, 1, 2, 3, 4, 5] comps=5 extra=0
bước 2: dây 0-2 | find=1,2 | gộp | parent=[1, 2, 2, 3, 4, 5] comps=4 extra=0
bước 3: dây 0-3 | find=2,3 | gộp | parent=[2, 2, 3, 3, 4, 5] comps=3 extra=0
bước 4: dây 1-2 | find=3,3 | dư  | parent=[2, 3, 3, 3, 4, 5] comps=3 extra=1
bước 5: dây 1-3 | find=3,3 | dư  | parent=[2, 3, 3, 3, 4, 5] comps=3 extra=2
=> đáp án 2


2

Trace DFS trên **Ví dụ 3** (để thấy cách đếm cụm, bỏ qua bước kiểm tra số dây):

In [17]:
def trace_dfs_components(n, connections):
    graph = [[] for _ in range(n)]
    for a, b in connections:
        graph[a].append(b)
        graph[b].append(a)
    seen = [False] * n
    comps = 0
    for s in range(n):
        if seen[s]:
            print(f"s={s}: đã thăm, bỏ qua")
            continue
        comps += 1
        seen[s] = True
        stack = [s]
        print(f"s={s}: cụm mới #{comps}")
        while stack:
            u = stack.pop()
            new = [v for v in graph[u] if not seen[v]]
            for v in new:
                seen[v] = True
                stack.append(v)
            print(f"   pop {u}, đẩy {new}, stack={stack}")
    print(f"số cụm = {comps}, số dây = {len(connections)}, cần {n-1}")
    return comps

trace_dfs_components(6, [[0, 1], [0, 2], [0, 3], [1, 2]])

s=0: cụm mới #1
   pop 0, đẩy [1, 2, 3], stack=[1, 2, 3]
   pop 3, đẩy [], stack=[1, 2]
   pop 2, đẩy [], stack=[1]
   pop 1, đẩy [], stack=[]
s=1: đã thăm, bỏ qua
s=2: đã thăm, bỏ qua
s=3: đã thăm, bỏ qua
s=4: cụm mới #2
   pop 4, đẩy [], stack=[]
s=5: cụm mới #3
   pop 5, đẩy [], stack=[]
số cụm = 3, số dây = 4, cần 5


3

## 9. Code nộp LeetCode

Bản chọn nộp: kiểm tra số dây + Union-Find (ngắn, nhanh, ít bộ nhớ).

In [18]:
class Solution:
    def makeConnected(self, n: int, connections: list[list[int]]) -> int:
        if len(connections) < n - 1:      # không đủ dây cho một cây khung
            return -1
        parent = list(range(n))

        def find(x: int) -> int:
            while parent[x] != x:
                parent[x] = parent[parent[x]]   # nén đường đi
                x = parent[x]
            return x

        comps = n
        for a, b in connections:
            ra, rb = find(a), find(b)
            if ra != rb:
                parent[ra] = rb
                comps -= 1
        return comps - 1

for n, conns, want in examples:
    got = Solution().makeConnected(n, conns)
    print(got, "| mong đợi", want, "| OK" if got == want else "| SAI")

1 | mong đợi 1 | OK
2 | mong đợi 2 | OK
-1 | mong đợi -1 | OK


## 10. Vì sao thuật toán đúng

**Cận dưới (cần ít nhất `c − 1`):** một thao tác thêm đúng 1 cạnh mới (và bỏ 1 cạnh). Thêm 1 cạnh chỉ có thể nối **hai** cụm thành một ⇒ số cụm giảm nhiều nhất 1; bỏ cạnh không bao giờ làm giảm số cụm. Từ `c` cụm về 1 cụm ⇒ cần **≥ `c − 1`** thao tác.

**Không thể khi `m < n − 1`:** mọi đồ thị liên thông `n` đỉnh chứa một cây khung với `n − 1` cạnh. Số cạnh bất biến qua các thao tác ⇒ không bao giờ đạt.

**Cận trên (làm được với `c − 1` khi `m ≥ n − 1`):** chọn trong mỗi cụm một cây khung; tổng số cạnh của các cây khung là `n − c`. Các cạnh còn lại (`m − n + c ≥ c − 1` cạnh) là **cạnh dư**: bỏ chúng đi các cụm vẫn liên thông. Lấy lần lượt `c − 1` cạnh dư, mỗi lần cắm nối cụm chứa máy `0` với một cụm chưa nối ⇒ sau `c − 1` thao tác còn 1 cụm. Cặp được cắm luôn là "hai máy chưa nối trực tiếp" vì chúng ở hai cụm khác nhau ⇒ hợp lệ.

**Invariant của Union-Find:** sau khi xử lý `k` dây đầu, mỗi tập trong DSU đúng bằng một thành phần liên thông của đồ thị chỉ gồm `k` dây đó, và `comps` = số tập ⇒ cuối cùng `comps = c`.

## 11. Phân tích Time / Space

Đặt `n` = số máy, `m = len(connections)`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| Brute force BFS trạng thái | `O(C(n(n−1)/2, m) · m · n²)` | số trạng thái | chỉ cho `n ≤ 5` |
| DFS (ngăn xếp) | `O(n + m)` | `O(n + m)` | danh sách kề + `seen` |
| BFS (deque) | `O(n + m)` | `O(n + m)` | như DFS |
| Union-Find | `O(n + m·α(n))` | `O(n)` | không cần danh sách kề |

Với `n, m ≤ 10^5`, cả ba cách tuyến tính đều chạy dưới vài chục mili-giây. Kiểm tra `m < n − 1` ở đầu là `O(1)` và giúp thoát sớm.

## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [19]:
solvers = {
    "dfs": make_connected_dfs,
    "bfs": make_connected_bfs,
    "uf": make_connected_uf,
    "submit": lambda n, c: Solution().makeConnected(n, c),
    "submit_dfs": lambda n, c: SolutionDFS().makeConnected(n, c),
    "submit_bfs": lambda n, c: SolutionBFS().makeConnected(n, c),
}
cases = examples + [
    (1, [], 0),                                  # một máy, không dây (ngoài ràng buộc m>=1 nhưng nên an toàn)
    (2, [[0, 1]], 0),                            # đã liên thông
    (3, [[0, 1]], -1),                           # thiếu dây
    (4, [[0, 1], [2, 3], [0, 2]], 0),            # đường thẳng, đúng n-1 dây
    (5, [[0, 1], [1, 2], [2, 0], [3, 4]], 1),    # m = n-1, 2 cụm
    (5, [[0, 1], [0, 2], [0, 3], [1, 2], [1, 3], [2, 3]], 1),  # K4 + 1 đỉnh lẻ
]
for n, conns, want in cases:
    for name, f in solvers.items():
        assert f(n, conns) == want, (name, n, conns)
print("Tất cả", len(cases), "case qua cho", len(solvers), "lời giải")

Tất cả 9 case qua cho 6 lời giải


### 12.2 Đối chiếu brute force (n nhỏ, seed cố định)

Sinh ngẫu nhiên đồ thị `n ∈ [2, 5]` với số dây hợp lệ, so sánh với brute force BFS trạng thái — đây là kiểm chứng **độc lập** cho công thức `c − 1`.

In [20]:
rng = random.Random(1319)
t0 = time.time()
cnt = 0
for _ in range(300):
    n = rng.randint(2, 5)
    pairs = list(combinations(range(n), 2))
    m = rng.randint(1, len(pairs))
    conns = [list(p) for p in rng.sample(pairs, m)]
    want = make_connected_brute(n, conns)
    for name, f in solvers.items():
        assert f(n, conns) == want, (name, n, conns, want)
    cnt += 1
print(f"{cnt} đồ thị ngẫu nhiên khớp brute force ({time.time() - t0:.2f}s)")

300 đồ thị ngẫu nhiên khớp brute force (0.02s)


### 12.3 Đối chiếu chéo các cách nhanh với `n` lớn hơn

In [21]:
rng = random.Random(7)
for _ in range(200):
    n = rng.randint(1, 60)
    pairs = list(combinations(range(n), 2))
    if not pairs:
        continue
    m = rng.randint(1, min(len(pairs), 80))
    conns = [list(p) for p in rng.sample(pairs, m)]
    rng.shuffle(conns)
    conns = [c[::-1] if rng.random() < 0.5 else c for c in conns]   # đảo chiều ngẫu nhiên
    res = {name: f(n, conns) for name, f in solvers.items()}
    assert len(set(res.values())) == 1, (n, conns, res)
print("200 đồ thị: các lời giải cho cùng kết quả")

200 đồ thị: các lời giải cho cùng kết quả


### 12.4 Giới hạn `n = m = 10^5`

In [22]:
rng = random.Random(99)
n = 10**5
conns = []
seen_pairs = set()
while len(conns) < 10**5:
    a, b = rng.randrange(n), rng.randrange(n)
    if a != b and (min(a, b), max(a, b)) not in seen_pairs:
        seen_pairs.add((min(a, b), max(a, b)))
        conns.append([a, b])
for name in ["dfs", "bfs", "uf", "submit"]:
    t0 = time.time()
    r = solvers[name](n, conns)
    print(f"{name:6}: {r}  ({time.time() - t0:.3f}s)")
chain = [[i, i + 1] for i in range(n - 1)]
print("dây chuyền 10^5 đỉnh:", Solution().makeConnected(n, chain))

dfs   : 16129  (0.096s)
bfs   : 16129  (0.118s)
uf    : 16129  (0.063s)


submit: 16129  (0.063s)
dây chuyền 10^5 đỉnh: 0


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên kiểm tra số dây, chỉ trả `comps − 1`

In [23]:
def wrong_no_check(n, connections):
    graph = [[] for _ in range(n)]
    for a, b in connections:
        graph[a].append(b)
        graph[b].append(a)
    seen = [False] * n
    comps = 0
    for s in range(n):
        if not seen[s]:
            comps += 1
            dfs_order(graph, s, seen)
    return comps - 1

n, conns, want = examples[2]
print("sai:", wrong_no_check(n, conns), "| đúng:", Solution().makeConnected(n, conns), "| mong đợi", want)

sai: 2 | đúng: -1 | mong đợi -1


**Vì sao sai:** Ví dụ 3 có 3 cụm nên trả `2`, nhưng chỉ có 4 dây cho 6 máy — dù di chuyển thế nào cũng không đủ. Số cụm chỉ cho cận dưới; phải kiểm tra thêm có đủ dây dư hay không. **Sửa:** thêm `if len(connections) < n - 1: return -1` ở đầu (như `Solution`).

### Lỗi 2 — DFS đệ quy quá sâu (RecursionError)

In [24]:
def wrong_recursive(n, connections):
    if len(connections) < n - 1:
        return -1
    graph = [[] for _ in range(n)]
    for a, b in connections:
        graph[a].append(b)
        graph[b].append(a)
    seen = [False] * n

    def dfs(u):
        seen[u] = True
        for v in graph[u]:
            if not seen[v]:
                dfs(v)

    comps = 0
    for s in range(n):
        if not seen[s]:
            comps += 1
            dfs(s)
    return comps - 1

n = 5000
chain = [[i, i + 1] for i in range(n - 1)]
try:
    print("đệ quy:", wrong_recursive(n, chain))
except RecursionError as e:
    print("đệ quy văng lỗi:", type(e).__name__, "| giới hạn đệ quy =", sys.getrecursionlimit())
print("ngăn xếp:", make_connected_dfs(n, chain))

đệ quy văng lỗi: RecursionError | giới hạn đệ quy = 1000
ngăn xếp: 0


**Vì sao sai:** đồ thị dạng **dây chuyền** làm DFS đệ quy sâu bằng số đỉnh; Python mặc định chỉ cho ~1000 tầng. Với `n = 10^5` chắc chắn vỡ. **Sửa:** dùng ngăn xếp tường minh (Cách 2), BFS, hoặc Union-Find. (Tăng `sys.setrecursionlimit` là chữa cháy, có thể làm sập tiến trình vì tràn stack C.)

### Lỗi 3 — `union` gán cha cho đỉnh, không phải cho gốc

In [25]:
def wrong_union(n, connections):
    if len(connections) < n - 1:
        return -1
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            x = parent[x]
        return x

    for a, b in connections:
        if find(a) != find(b):
            parent[a] = b          # SAI: phải là parent[find(a)] = find(b)
    roots = sum(1 for i in range(n) if find(i) == i)
    return roots - 1

conns = [[0, 1], [0, 2]]
print("sai:", wrong_union(3, conns), "| đúng:", Solution().makeConnected(3, conns))

sai: 1 | đúng: 0


**Vì sao sai:** sau `0-1`: `parent[0] = 1`. Xử lý `0-2`: `find(0) = 1 ≠ 2` ⇒ gán `parent[0] = 2` — **ghi đè** liên kết cũ, đỉnh `1` bị "bỏ rơi" thành gốc riêng. Kết quả đếm 2 gốc (`1` và `2`) ⇒ trả `1` dù đồ thị `1-0-2` đã liên thông (đáp án đúng `0`). **Sửa:** luôn gộp **gốc với gốc**: `ra, rb = find(a), find(b); parent[ra] = rb`.

### Lỗi 4 — Tính số cụm bằng `n − m`

In [26]:
def wrong_n_minus_m(n, connections):
    if len(connections) < n - 1:
        return -1
    comps = n - len(connections)     # chỉ đúng khi đồ thị là rừng (không chu trình)
    return max(comps, 1) - 1

for n, conns, want in examples[:2]:
    print(f"n={n}: sai={wrong_n_minus_m(n, conns)}, đúng={Solution().makeConnected(n, conns)}")

n=4: sai=0, đúng=1
n=6: sai=0, đúng=2


**Vì sao sai:** công thức `c = n − m` chỉ đúng khi **không có chu trình**. Ví dụ 1 có chu trình `0-1-2` ⇒ `n − m = 1` nhưng thật ra có 2 cụm. Mỗi cạnh tạo chu trình **không** giảm số cụm. **Sửa:** đếm cụm thật bằng DFS/BFS/DSU.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề nói "**nối tất cả**", "**liên lạc được với nhau**", "mạng", "đảo", "nhóm bạn bè" ⇒ nghĩ ngay **thành phần liên thông**.
- "Số thao tác ít nhất để nối" thường = **số cụm − 1** (mỗi thao tác nối được 2 cụm).
- "Có đủ tài nguyên không?" ⇒ tìm **bất biến** (ở đây: số dây không đổi) và **điều kiện cần** (cây khung cần `n − 1` cạnh).
- Đồ thị cho dưới dạng **danh sách cạnh** + chỉ cần biết ai cùng nhóm ⇒ **Union-Find** thường gọn nhất.
- Python: DFS lớn ⇒ dùng **ngăn xếp**, đừng đệ quy.
- Muốn chắc công thức? Viết **brute force trạng thái** cho `n ≤ 5` và đối chiếu ngẫu nhiên.

## 15. Biến thể của bài

1. **Trả về danh sách thao tác** (dây nào rút, cắm vào đâu) — xem Bài tập 5.
2. **LeetCode 547 — Number of Provinces:** đếm cụm từ ma trận kề.
3. **LeetCode 323 — Number of Connected Components in an Undirected Graph** (Premium): đếm cụm thuần tuý.
4. **LeetCode 1584 — Min Cost to Connect All Points / Kruskal:** khi cạnh có trọng số, nối tất cả với chi phí nhỏ nhất ⇒ cây khung nhỏ nhất (MST) dùng DSU.
5. **LeetCode 684 — Redundant Connection:** tìm cạnh dư (cạnh mà `find(a) == find(b)`).
6. Biến thể tự chế: mỗi thao tác được **thêm** dây mới (không cần rút) ⇒ đáp án luôn `c − 1`, không bao giờ `-1`.

In [27]:
def count_provinces(is_connected):
    """LeetCode 547: đếm cụm từ ma trận kề n x n."""
    n = len(is_connected)
    d = DSU(n)
    for i in range(n):
        for j in range(i + 1, n):
            if is_connected[i][j]:
                d.union(i, j)
    return d.count

def redundant_edges(n, connections):
    """Liệt kê các cạnh dư theo thứ tự duyệt (biến thể LeetCode 684)."""
    d = DSU(n)
    return [e for e in connections if not d.union(*e)]

print(count_provinces([[1, 1, 0], [1, 1, 0], [0, 0, 1]]))
print(redundant_edges(6, examples[1][1]))

2
[[1, 2], [1, 3]]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force trạng thái | BFS trên mọi cách rút/cắm dây | bùng nổ tổ hợp | bùng nổ | `n ≤ 5`, kiểm chứng công thức |
| 2. DFS ngăn xếp | đếm cụm, kiểm tra `m ≥ n−1` | `O(n + m)` | `O(n + m)` | quen thuộc, dễ mở rộng (liệt kê cụm) |
| 3. BFS deque | như DFS, loang theo lớp | `O(n + m)` | `O(n + m)` | khi cần khoảng cách/lớp |
| 4. Union-Find | gộp tập theo từng cạnh | `O(n + m·α(n))` | `O(n)` | cạnh dạng danh sách, cần đếm cạnh dư |

## 17. Bài tập vận dụng

### Bài 1 (đoán output)
`Solution().makeConnected(5, [[0,1],[1,2],[2,0],[3,4]])` trả về gì? Còn `makeConnected(5, [[0,1],[1,2],[3,4]])`?

### Bài 2 (tính tay)
`n = 7`, `connections = [[0,1],[1,2],[2,0],[3,4],[4,5],[5,3],[3,6]]`. Có bao nhiêu cụm, bao nhiêu dây dư, đáp án là gì?

### Bài 3 (điền TODO)
Hoàn thiện hàm `find` (có nén đường đi) và phần gộp trong `count_components_todo`.

### Bài 4 (tự viết lại)
Viết lại lời giải bằng **BFS** mà không nhìn code, đặt tên `make_connected_ex4(n, connections)`.

### Bài 5 (mini challenge)
Viết `plan_operations(n, connections)` trả về **danh sách thao tác** `[(dây_rút, dây_cắm), ...]` với số thao tác tối thiểu (hoặc `None` nếu không thể). Viết thêm hàm kiểm tra: áp dụng lần lượt các thao tác (dây rút phải đang tồn tại, dây cắm phải chưa tồn tại) và cuối cùng mạng liên thông.

In [28]:
# Bài 3 — điền TODO (cell này chỉ định nghĩa hàm, không gọi)
def count_components_todo(n, connections):
    parent = list(range(n))

    def find(x):
        # TODO: leo lên gốc, có nén đường đi
        pass

    comps = n
    for a, b in connections:
        # TODO: nếu hai gốc khác nhau thì gộp và giảm comps
        pass
    return comps

In [29]:
# Bài 4 & 5 — khung để bạn tự viết (không gọi)
def make_connected_ex4(n, connections):
    # TODO: BFS đếm cụm + kiểm tra số dây
    pass

def plan_operations_todo(n, connections):
    # TODO: tìm cạnh dư bằng DSU, đại diện của mỗi cụm, rồi ghép
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1:** (1) Đếm số dây so với `n − 1`. (2) Đếm số cụm. (3) Lần 1: 4 dây = `n − 1`, 2 cụm ⇒ ?; lần 2: 3 dây < 4 ⇒ ?

**Bài 2:** (1) Vẽ hai cụm tam giác. (2) Cụm `{3,4,5,6}` có 4 dây cho 4 đỉnh ⇒ dư 1; cụm `{0,1,2}` dư 1. (3) 2 cụm ⇒ cần 1 thao tác, có 2 dây dư ⇒ đáp án 1.

**Bài 3:** (1) `while parent[x] != x`. (2) Nén: `parent[x] = parent[parent[x]]` rồi `x = parent[x]`. (3) Gộp: `ra, rb = find(a), find(b)`; nếu khác thì `parent[ra] = rb; comps -= 1`.

**Bài 4:** (1) Kiểm tra số dây trước. (2) Danh sách kề hai chiều + `seen`. (3) Mỗi lần gặp đỉnh chưa thăm ⇒ `comps += 1` rồi BFS bằng `deque`.

**Bài 5:** (1) Dùng DSU: cạnh có `find(a) == find(b)` là cạnh dư. (2) Sau khi duyệt hết, lấy một đại diện (gốc) cho mỗi cụm. (3) Thao tác thứ `i`: rút cạnh dư thứ `i`, cắm `(đại_diện[0], đại_diện[i+1])`.

## 19. Đáp án tham khảo

In [30]:
# Bài 1
print("Bài 1:", Solution().makeConnected(5, [[0, 1], [1, 2], [2, 0], [3, 4]]),
      Solution().makeConnected(5, [[0, 1], [1, 2], [3, 4]]))
assert Solution().makeConnected(5, [[0, 1], [1, 2], [2, 0], [3, 4]]) == 1
assert Solution().makeConnected(5, [[0, 1], [1, 2], [3, 4]]) == -1

# Bài 2
conns2 = [[0, 1], [1, 2], [2, 0], [3, 4], [4, 5], [5, 3], [3, 6]]
print("Bài 2: cụm =", components_by_hand(7, conns2), "| dư =", redundant_edges(7, conns2))
assert Solution().makeConnected(7, conns2) == 1

# Bài 3
def count_components_sol(n, connections):
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    comps = n
    for a, b in connections:
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[ra] = rb
            comps -= 1
    return comps

assert count_components_sol(6, examples[1][1]) == 3
assert count_components_sol(4, examples[0][1]) == 2

# Bài 4
def make_connected_ex4_sol(n, connections):
    if len(connections) < n - 1:
        return -1
    graph = [[] for _ in range(n)]
    for a, b in connections:
        graph[a].append(b)
        graph[b].append(a)
    seen = [False] * n
    comps = 0
    for s in range(n):
        if not seen[s]:
            comps += 1
            seen[s] = True
            q = deque([s])
            while q:
                u = q.popleft()
                for v in graph[u]:
                    if not seen[v]:
                        seen[v] = True
                        q.append(v)
    return comps - 1

for n, conns, want in cases:
    assert make_connected_ex4_sol(n, conns) == want
print("Bài 1–4: qua")

Bài 1: 1 -1
Bài 2: cụm = [[0, 1, 2], [3, 4, 5, 6]] | dư = [[2, 0], [5, 3]]
Bài 1–4: qua


In [31]:
# Bài 5
def plan_operations(n, connections):
    if len(connections) < n - 1:
        return None
    d = DSU(n)
    extra = [tuple(e) for e in connections if not d.union(*e)]
    reps = sorted({d.find(i) for i in range(n)})
    return [(extra[i], (reps[0], reps[i + 1])) for i in range(len(reps) - 1)]

def check_plan(n, connections, plan):
    edges = {tuple(sorted(e)) for e in connections}
    for removed, added in plan:
        removed, added = tuple(sorted(removed)), tuple(sorted(added))
        if removed not in edges or added in edges or added[0] == added[1]:
            return False
        edges.remove(removed)
        edges.add(added)
    return is_connected(n, edges)

print("Bài 5 ví dụ 2:", plan_operations(*examples[1][:2]))
rng = random.Random(55)
for _ in range(300):
    n = rng.randint(1, 30)
    pairs = list(combinations(range(n), 2))
    if not pairs:
        continue
    m = rng.randint(1, min(len(pairs), 40))
    conns = [list(p) for p in rng.sample(pairs, m)]
    want = Solution().makeConnected(n, conns)
    plan = plan_operations(n, conns)
    if want == -1:
        assert plan is None
    else:
        assert len(plan) == want and check_plan(n, conns, plan)
print("Bài 5: 300 test ngẫu nhiên — kế hoạch hợp lệ và tối thiểu")

Bài 5 ví dụ 2: [((1, 2), (0, 4)), ((1, 3), (0, 5))]
Bài 5: 300 test ngẫu nhiên — kế hoạch hợp lệ và tối thiểu


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template Union-Find đếm cụm:**

```python
if len(edges) < n - 1:
    return -1
parent = list(range(n))
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x
comps = n
for a, b in edges:
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[ra] = rb
        comps -= 1
return comps - 1
```

**Công thức nhớ:** `m < n − 1 ⇒ -1`, ngược lại `đáp án = số cụm − 1`.

**Checklist trước khi submit:**
- [ ] Đã kiểm tra `len(connections) < n - 1` ⇒ `-1`.
- [ ] Danh sách kề thêm **hai chiều** (nếu dùng DFS/BFS).
- [ ] DFS viết bằng **ngăn xếp** (không đệ quy sâu).
- [ ] `union` gộp **gốc với gốc**.
- [ ] Đỉnh cô lập (không có dây nào) vẫn được tính là một cụm — duyệt `range(n)`, không chỉ các đỉnh xuất hiện trong cạnh.
- [ ] Đúng chữ ký: `makeConnected(self, n: int, connections: list[list[int]]) -> int`.